In [23]:
import pandas as pd 

df = pd.read_csv(
    "../data/steam-200k.csv",
    header=None,
    names=["id_player","game","buy?","playtime","unused"]
)

owner_per_game = (
    df[df["buy?"] == "purchase"].groupby(by="game")["id_player"]
    .count()
    .reset_index(name="owners")
)
stat_per_game = (
    df[df["buy?"] == "play"].groupby(by="game")["playtime"]
    .agg(["sum","mean"])
    .round(2)
)
nb_player_per_game=(
    df[(df["buy?"] == "play") & (df["playtime"] > 0)] 
    .groupby(by="game")["id_player"]
    .count()
    .reset_index(name="player")
)
final_table = owner_per_game.merge(nb_player_per_game, left_on="game",right_on="game")
final_table = final_table.merge(stat_per_game, left_on="game",right_on="game")
final_table.columns=["Game","owner","players","total_playtime","mean_playtime"]
final_table.to_csv("../data/game_stat.csv",index=False)
final_table_playtime = (
    final_table.sort_values("total_playtime",ascending=False)
    .reset_index(drop=True)
    .head(n=10)
)
final_table_owner= (
    final_table.sort_values("owner",ascending=False)
    .reset_index(drop=True)
    .head(n=10)
)
final_table_mean = (
    final_table[final_table["players"] >= 50]
    .sort_values("mean_playtime",ascending=False)
    .reset_index(drop=True)
    .head(n=10)
)
#print("top 10 by owners\n",final_table_owner.to_string(index=False),"\n")
#print("top 10 by total_playtime\n",final_table_playtime.to_string(index=False),"\n")
#print("top 10 by mean_playtime\n",final_table_mean.to_string(index=False),"\n")

import pandas as pd
def recommend_popular_game(player_id,game_stat,n=10,metric="owner",min_players=50):
    purchase = df[df["buy?"] == "purchase"]
    owned_game = purchase[purchase["id_player"] == player_id]["game"]

    stat = game_stat[~game_stat["Game"].isin(owned_game)].copy()
    if(metric == "mean_playtime"):
        stat=stat[stat["players"] >= min_players]
    
    return(
        stat
        .sort_values(metric,ascending=False)
        .head(n)
    )

print(recommend_popular_game(128470551,final_table,10,"owner").to_string(index=False),"\n"
)
print(recommend_popular_game(221315846,final_table,10,"owner").to_string(index=False),"\n"
)
print(recommend_popular_game(99096740,final_table,10,"owner").to_string(index=False)
)



                           Game  owner  players  total_playtime  mean_playtime
                         Dota 2   4841     4841        981684.6         202.79
                Team Fortress 2   2323     2323        173673.3          74.76
                       Unturned   1563     1069         16096.4          15.06
Counter-Strike Global Offensive   1412     1377        322771.6         234.40
         Half-Life 2 Lost Coast    981      177           184.4           1.04
          Counter-Strike Source    978      715         96075.5         134.37
                  Left 4 Dead 2    951      801         33596.7          41.94
                 Counter-Strike    856      568        134261.1         236.38
                       Warframe    847      424         27074.6          63.86
         Half-Life 2 Deathmatch    823      198          3712.9          18.75 

                           Game  owner  players  total_playtime  mean_playtime
                       Unturned   1563     1069   

In [ ]:
import pandas as pd 
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_csv(
    "../data/steam-200k.csv",
    header=None,
    names=["id_player","game","buy?","playtime","unused"]
)

game_played = (
    df[df["buy?"] == "play"]
    .pivot_table(
        values="playtime",
        index="id_player",
        columns="game",
        fill_value=0
    )    
)
user_game_sparse = csr_matrix(game_played.values)
non_zero = (game_played != 0).sum().sum()
total = game_played.shape[0] * game_played.shape[1]
density = non_zero / total
sparsity = 1 - density

def player_similarity(player_id):
    player_index = game_played.index.get_loc(player_id)
    user_vector = user_game_sparse[player_index]
    similarities = cosine_similarity(user_vector,user_game_sparse)
    similarities = similarities.flatten()
    print(similarities.shape)
    similarities = similarities.argsort(similarities,False)
    
    return similarities
#print(f"Density: {density:.4%}")
#print(f"Sparsity: {sparsity:.4%}")
#print(user_game_sparse.shape)
#print(user_game_sparse)
#print(game_played.index[:5])
print(player_similarity(76767))

(11350,)


TypeError: only integer scalar arrays can be converted to a scalar index